In [1]:
######## LIBRARIES
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
import pandas as pd
import pydicom 
import scipy.stats as stats
from statsmodels.stats.weightstats import DescrStatsW
import tkinter as tk
from tkinter import filedialog, simpledialog
from tkinter.filedialog import askdirectory
######## PDF
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, KeepTogether
from reportlab.platypus import Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.enums import TA_CENTER

In [2]:
######## SEARCH WINDOW
def select_file(
        title="Select file",
        filetypes=(("All files", "*.*"),)
        ):
    root = tk.Tk()
    root.withdraw()
    file = filedialog.askopenfilename(
        title=title,
        initialdir="C:/",
        filetypes=filetypes
    )
    root.destroy()
    return file

In [3]:
######## LOG FILE (CSV) SELECTION WINDOW
def select_csvs():
    root = tk.Tk()
    root.withdraw()
    
    # Number of log files (CSV)
    num_csv = simpledialog.askinteger(
        "Log files",
        "Number of log files in this session",
        minvalue = 1,  # Min. number of CSV
        maxvalue = 5   #Max. number of CSV
    )
    if num_csv is None:
        print("ERROR.")
        root.destroy()
        return []
    files_csv = []
    for i in range(num_csv):
        root_csv = select_file(
            title=f"Select log file (csv) number {i+1} from this session",
            filetypes=(("CSV files", "*.csv"),)
        )
        if root_csv:
            files_csv.append(root_csv)
            
    root.destroy()
    return files_csv

In [4]:
######## CONCATENATE LOG FILES (CSV)
def concat_csvs(root_csv):
    if not root_csv:
        print("No log file was selected")
        return None

    # Read all DataFrames in a list
    dataframes = [pd.read_csv(root, skiprows=9) for root in root_csv]

    # If there is only one log file, open it directly
    if len(dataframes) == 1:
        print("Only one log file was selected")
        return dataframes[0]

    # Column name
    col_time = "Time (s)"
    col_cp = "Control point/Actual Value (None)"

    # Continuity between archives
    for i in range(1, len(dataframes)):
        last_time_record = dataframes[i-1][col_time].iloc[-1]
        last_cp_record = dataframes[i-1][col_cp].iloc[-1]

        dataframes[i][col_time] += last_time_record
        dataframes[i][col_cp] += last_cp_record

    # Concatenate all DataFrames
    df_concat = pd.concat(dataframes, ignore_index=True)
    return df_concat

In [5]:
######## DICOM DATA (DCM)
def dcm_data(root_dcm): 
    dcm = pydicom.dcmread(
        root_dcm, 
        force=True
        )
    rows = []
    DCM_Dose = 0
    DCM_CP = 1
    # Info of treatment & pacient
    P_Name = dcm.PatientName
    P_ID = dcm.PatientID
    Intent = dcm.PlanIntent
    
    # Treatment target
    if hasattr(dcm, 'DoseReferenceSequence'):
        Target = dcm.DoseReferenceSequence[0].DoseReferenceDescription

    # Info per beam
    for beam in dcm.BeamSequence:
        BN = beam.BeamNumber

        # Total dose per beam
        DCM_BD = None
        for fg in dcm.FractionGroupSequence:
            for r_beam in fg.ReferencedBeamSequence:
                if r_beam.ReferencedBeamNumber == BN:
                    DCM_BD = r_beam.BeamMeterset
                    break
            if DCM_BD is not None:
                break

        # Total dose
        DCM_Dose += DCM_BD
        
        # Gantry angle
        Gantry = beam.ControlPointSequence[0].GantryAngle

        # Info per control point
        for i, cp in enumerate(beam.ControlPointSequence):
            row = {
                "Beam Number": BN,
                "Control Point": DCM_CP,
                "Beam Dose": DCM_BD,
                "Total Dose [MU]": DCM_Dose,
                "Gantry Angle": Gantry,
            }

            # Dose per cp
            MW = cp.CumulativeMetersetWeight
            if i == 0:
                MU_cp = 0
            else:
                MW_prev = beam.ControlPointSequence[i-1].CumulativeMetersetWeight
                MU_cp = (MW - MW_prev) * DCM_BD
            row["Dose per CP"] = MU_cp

            # Info Jaws & MLC
            if hasattr(cp, "BeamLimitingDevicePositionSequence"):
                for bld in cp.BeamLimitingDevicePositionSequence:
                    device_type = bld.RTBeamLimitingDeviceType
                    pos = np.array(bld.LeafJawPositions)

                    # Jaws
                    if device_type in ("ASYMX", "ASYMY") and pos.size == 2:
                        row["JAW X2"] = float(pos[0])
                        row["JAW X1"] = float(pos[1])

                    # MLCs
                    if device_type in ("MLCX", "MLCY") and pos.size == 160:
                        DCM_MLC_L = pos[:80]
                        DCM_MLC_R = pos[80:]
                        for j in range(80):
                            row[f"MLC Y1: {j+1}"] = DCM_MLC_L[j]
                        for j in range(80):
                            row[f"MLC Y2: {j+1}"] = DCM_MLC_R[j]

            rows.append(row)
            DCM_CP += 1

    # DataFrame with DCM info
    df_DCM = pd.DataFrame(rows).iloc[:-1]
    
    # Info per column
    DCM_CP_DF = df_DCM["Control Point"].tolist()
    DCM_TD = df_DCM["Total Dose [MU]"]
    DCM_NH = df_DCM["Beam Number"]
    DCM_BD = df_DCM["Beam Dose"]
    DCM_DPC = df_DCM["Dose per CP"]
    DCM_GANTRY = df_DCM["Gantry Angle"]
    DCM_JAW_X1 = df_DCM["JAW X1"]
    DCM_JAW_X2 = df_DCM["JAW X2"]


    # Left MLC
    DCM_Y1_C = df_DCM.loc[:, "MLC Y1: 1":"MLC Y1: 80"].iloc[:, ::-1]
    # Rigth MLC
    DCM_Y2_C = df_DCM.loc[:, "MLC Y2: 1":"MLC Y2: 80"].iloc[:, ::-1]

    # Area per CP
    DCM_ACP = ((DCM_Y2_C.values - DCM_Y1_C.values)*5).sum(axis=1).tolist()
    df_DCM["Area per CP"] = DCM_ACP
    
    # Dictionary with all data
    data = {
        "Control Point": DCM_CP_DF,
        "Beam Number": DCM_NH,
        "Total Dose": DCM_TD,
        "Dose per Beam": DCM_BD,
        "Dose per CP": DCM_DPC,
        "Area per CP": DCM_ACP,
        "Gantry Angle": DCM_GANTRY,
        "Jaw X1": DCM_JAW_X1,
        "Jaw X2": DCM_JAW_X2,
        "MLC Y1": DCM_Y1_C,
        "MLC Y2": DCM_Y2_C
    }

    patient = {
        "Patient Name": P_Name,
        "Patient ID": P_ID,
        "Target": Target,
        "Intent": Intent
    }

    return df_DCM, data, patient

In [6]:
######## LOG FILE (CSV) DATA
def csv_data(df_CSV):
    # Time
    Time_c = "Time (s)"
    Time = df_CSV[Time_c]

    # Control Points
    CSV_CP_C = "Control point/Actual Value (None)"
    CSV_CP = df_CSV[CSV_CP_C].unique().tolist()

    # Linac state
    LS_C = "Linac State/Actual Value (None)"
    LS = df_CSV[LS_C]

    # Filter by linac state
    F_RO = df_CSV[LS == "Radiation On"] 
    F_MO = df_CSV[LS == "Move Only"]     

    # Dose
    DR_C = "Actual Dose Rate/Actual Value (Mu/min)"
    DR_d = df_CSV[DR_C]
    F_DR = df_CSV[DR_d != 0]  

    AD_C = "Step Dose/Actual Value (Mu)"
    CSV_D_CP = F_RO.groupby(CSV_CP_C)[AD_C].max()
    CSV_TD = CSV_D_CP.sum()

    # Gantry
    CSV_GTY_C = "Step Gantry/Scaled Actual (deg)"
    GANTRY = F_RO.groupby(CSV_CP_C)[CSV_GTY_C].mean()
    CSV_GANTRY = GANTRY.where(GANTRY > 0, GANTRY + 360)

    # Jaws
    JAW_X1 = "X1 Diaphragm/Scaled Actual (mm)"
    JAW_X2 = "X2 Diaphragm/Scaled Actual (mm)"
    CSV_JAWS_1 = F_RO.groupby(CSV_CP_C)[JAW_X1].mean()
    CSV_JAWS_2 = (-1) * F_RO.groupby(CSV_CP_C)[JAW_X2].mean()

    # Left MLC
    I_CY1 = "Y1 Leaf 1/Scaled Actual (mm)"
    F_CY1 = "Y1 Leaf 80/Scaled Actual (mm)"
    I_Y1 = df_CSV.columns.get_loc(I_CY1)
    F_Y1 = df_CSV.columns.get_loc(F_CY1)
    MLC_I = df_CSV.columns[I_Y1:F_Y1 + 1]

    # Right MLC
    I_CY2 = "Y2 Leaf 1/Scaled Actual (mm)"
    F_CY2 = "Y2 Leaf 80/Scaled Actual (mm)"
    I_Y2 = df_CSV.columns.get_loc(I_CY2)
    F_Y2 = df_CSV.columns.get_loc(F_CY2)
    MLC_D = df_CSV.columns[I_Y2:F_Y2 + 1]

    # Group by CP (Radiation On)
    Y1 = F_RO.groupby(CSV_CP_C)[MLC_I].mean()
    CSV_Y1_MLC = (-1) * Y1.where(Y1 > -110, Y1 + 33.5)
    Y2 = F_RO.groupby(CSV_CP_C)[MLC_D].mean()
    CSV_Y2_MLC = Y2.where(Y2 < 110, Y2 - 33.5)

    # Group by CP (Move Only)
    Y1_MOVE = F_MO[MLC_I]
    Y2_MOVE = F_MO[MLC_D]
    TIME_MOVE = F_MO[Time_c]

    # Effective time 
    Time_E_min = F_DR.groupby(CSV_CP_C)[Time_c].min()
    Time_E_max = F_DR.groupby(CSV_CP_C)[Time_c].max()
    Time_Ef = (Time_E_max - Time_E_min).sum()  # min
    Total_Time = Time.max()
    P_Ef_Time = (Time_Ef * 100) / Total_Time

    # Dictionary with all data
    data_CSV = {
        "Control Point": CSV_CP,
        "Dose per CP": CSV_D_CP,
        "Total Dose [MU]": CSV_TD,
        "Total Time": round(Total_Time, 2),
        "Effective Time": round(Time_Ef, 2),
        "Percentage of Effective Time [%]": round(P_Ef_Time, 2),
        "Time in movement": TIME_MOVE,
        "Gantry Angle": CSV_GANTRY,
        "Jaw X1": CSV_JAWS_1,
        "Jaw X2": CSV_JAWS_2,
        "MLC Y1": CSV_Y1_MLC,
        "MLC Y2": CSV_Y2_MLC,
        "MLC Y1 MOV": Y1_MOVE,
        "MLC Y2 MOV": Y2_MOVE,
    }

    return df_CSV, data_CSV

In [7]:
######## SELECT DCM FILE
R_DCM = select_file(
    title="Select DICOM RT PLAN file", 
    filetypes=(("DICOM files", "*.dcm"),)
    )

In [8]:
######## EXTRACT DATA FROM A DICOM RT PLAN
df_DCM, data_DCM, info_Treat = dcm_data(R_DCM)

# Patient data
Patient = info_Treat["Patient Name"]
ID = info_Treat["Patient ID"]
Target = info_Treat["Target"]
Intento = info_Treat["Intent"]
Date = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

# Geometry & Treatment data
DCM_CP_DF = data_DCM["Control Point"]
DCM_NH = data_DCM["Beam Number"]
DCM_TD = data_DCM["Total Dose"]
DCM_DH = data_DCM["Dose per Beam"]
DCM_DPC = data_DCM["Dose per CP"]
DCM_GANTRY = data_DCM["Gantry Angle"]
DCM_Y1_MLC = data_DCM["MLC Y1"]
DCM_Y2_MLC = data_DCM["MLC Y2"]
DCM_JAW_X1 = data_DCM["Jaw X1"]
DCM_JAW_X2 = data_DCM["Jaw X2"]

In [9]:
######## SELECT LOG FILES (CSV)
root_csv = select_csvs()
df_CSV_concat = concat_csvs(root_csv)

Only one log file was selected


In [10]:
######## EXTRACT DATA FROM LOG FILES (CSV)
df_CSV, data_CSV = csv_data(df_CSV_concat)

# Data:
CSV_CP_DF = data_CSV["Control Point"]
CSV_TD = data_CSV["Total Dose [MU]"]
CSV_D_CP = data_CSV["Dose per CP"]
CSV_GANTRY = data_CSV["Gantry Angle"]
CSV_Y1_MLC = data_CSV["MLC Y1"]
CSV_Y2_MLC = data_CSV["MLC Y2"]
Y1_MOVE = data_CSV["MLC Y1 MOV"]
Y2_MOVE = data_CSV["MLC Y2 MOV"]
CSV_JAW_X1 = data_CSV["Jaw X1"]
CSV_JAW_X2 = data_CSV["Jaw X2"]
CSV_TIME_TOTAL = data_CSV["Total Time"]
CSV_TIME_EF = data_CSV["Effective Time"]
CSV_P_TOTAL = data_CSV["Percentage of Effective Time [%]"]
TIME_MOV = data_CSV["Time in movement"]

In [11]:
######## INTERSECT CP (RADIATION ON)
def CPC(CP_DCM, MLC_CSV):
    C_CP = np.intersect1d(CP_DCM, MLC_CSV.index)
    return C_CP

In [12]:
####### INTERSECTION OF CONTROL POINTS 
CP_common = CPC(DCM_CP_DF, CSV_Y1_MLC)

In [13]:
######## FILTER MLC BY CP (RADIATION ON)

# DICOM
DCM_Y1 = DCM_Y1_MLC.iloc[CP_common - 1]
DCM_Y2 = DCM_Y2_MLC.iloc[CP_common - 1]

# CSV
CSV_Y1 = CSV_Y1_MLC.loc[CP_common]
CSV_Y2 = CSV_Y2_MLC.loc[CP_common]

In [14]:
######## FILTER JAWS BY CP (RADIATION ON)

#DCM
DCM_X1 = DCM_JAW_X1.iloc[CP_common - 1]
DCM_X2 = DCM_JAW_X2.iloc[CP_common - 1]

#CSV
CSV_X1 = CSV_JAW_X1.loc[CP_common]
CSV_X2 = CSV_JAW_X2.loc[CP_common]

In [15]:
######## FILTER GANTRY ANGLE BY CP (RADIATION ON)

#DCM
DCM_GTY = DCM_GANTRY.iloc[CP_common - 1]

#CSV
CSV_GTY = CSV_GANTRY.loc[CP_common]

In [16]:
######### STATISTICS
def STAT(DIF):
    
    # Vector
    DIF = np.array(DIF).flatten()

    Vesp = 0
    t_stat, p_value = stats.ttest_1samp(DIF, Vesp)
    
    # STATS
    mean = np.mean(DIF)
    std = np.std(DIF, ddof=1)
    dsw = DescrStatsW(DIF)
    
    # CONFIDENCE INTERVAL
    alpha=0.05
    ci_low, ci_high = dsw.tconfint_mean(alpha=alpha)
    
    return {
        "Mean": mean,
        "SD": std,
        "T-Student": t_stat,
        "P-value": p_value,
        "ci": (ci_low, ci_high),
    }

In [17]:
######## DIFERENCE OF MU
def D_MU(D_DCM, D_CSV):
    root = tk.Tk()
    root.withdraw()
    
    Dif_D = D_DCM.max() - D_CSV
    E_PER = abs(round(Dif_D*100/D_DCM.max(), 2))
    messages_MU = []
    messages_MU.append(
        f"Monitor units DICOM: {round(D_DCM.max(),2)} MU"
        )
    messages_MU.append(
        f"Monitor units log file: {round(D_CSV,2)} MU"
        )
    messages_MU.append(
        f"Difference of monitor units: {round(Dif_D,2)} MU"
        )
    messages_MU.append(
        f"Percent difference: {E_PER} %"
        )
    return{
        "messages": messages_MU
    }

In [18]:
######## DIF. MU
E_D = D_MU(DCM_TD, CSV_TD)

In [19]:
######## DIF. MLC (DCM vs CSV)
def DMLC(MLC_DCM, MLC_CSV):
    DIF = MLC_DCM.values - MLC_CSV.values
    return DIF

In [20]:
######## DIF. MLC (BANK Y1 & Y2)
Dif_Y1 = DMLC(DCM_Y1, CSV_Y1)   #LEFT
Dif_Y2 = DMLC(DCM_Y2, CSV_Y2)   #RIGHT

In [21]:
######## ERROR MLC
def D_MLC(Dif_MLC, Y):
    N_L = Dif_MLC.shape[1] + 1
    N = np.arange(1, N_L)
    E_P_MLC = Dif_MLC.mean(axis=0)
    E_P_MLC_P = np.where(E_P_MLC > 1)[0] + 1 #POSITIVE
    E_P_MLC_N = np.where(E_P_MLC < -1)[0] + 1 #NEGATIVE
    # Statistic
    stat = STAT(Dif_MLC)
    # Scatter
    fig_name = f"Difference of the MLC bank {Y}.png"
    plt.figure(figsize=(15, 6))
    plt.scatter(N, E_P_MLC)
    plt.xlabel("Leaf number")
    plt.ylabel("Average difference [mm]")
    plt.title(f"Average difference per leaf of the MLC (bank {Y})")
    plt.axhline(1, color='g', linestyle='--')
    plt.axhline(0, color='black', linestyle='--')
    plt.axhline(-1, color='g', linestyle='--')
    plt.xticks(N)
    plt.xticks(rotation=90)
    plt.grid(True)
    plt.savefig(fig_name, dpi=300, bbox_inches="tight")
    plt.close()
    # Messages
    messages_MLC = []
    if len(E_P_MLC_P) > 0:
        messages_MLC.append(
            f"Leaf from banck {Y} with difference outside the limit > 1 mm: {E_P_MLC_P.tolist()}"
        )
    else:
        messages_MLC.append(
            f"No leaf from bank {Y} is outside the suggested limit > 1 mm"
        )

    if len(E_P_MLC_N) > 0:
        messages_MLC.append(
            f"Leaf from banck {Y} with difference outside the limit < -1 mm: {E_P_MLC_N.tolist()}"
        )
    else:
        messages_MLC.append(
            f"No leaf from bank {Y} is outside the suggested limit < -1 mm"
        )

    messages_MLC_std = []
    messages_MLC_std.append(f"Mean value: {stat['Mean']:.5f} mm")
    messages_MLC_std.append(f"SD: {stat['SD']:.5f} mm")
    messages_MLC_std.append(f"P-value: {stat['P-value']:.5e}")
    ci_low, ci_high = stat["ci"]
    messages_MLC_std.append(f"CI 95%: [{ci_low:.5f}, {ci_high:.5f}] mm")

    return {
        "image": fig_name,
        "messages_pos": messages_MLC,
        "messages_std": messages_MLC_std
    }

In [22]:
######## ERROR BANK Y1
E_Y1 = D_MLC(Dif_Y1, "Y1")

In [23]:
######## ERROR BANK Y2
E_Y2 = D_MLC(Dif_Y2, "Y2")

In [24]:
######## VIOLINPLOT WITHOUT OUTLIERS
def D_MLC_VIO(D_MLC, Y):
    N_L = D_MLC.shape[1]
    # DataFrame
    df_dif_MLC = pd.DataFrame(D_MLC, columns=[i+1 for i in range(N_L)])
    df_long = df_dif_MLC.melt(var_name="MLC", value_name="Error [mm]")

    # IQR (InterQuartile Range) 
    Q1 = df_long["Error [mm]"].quantile(0.05)    #QUARTILE 1
    Q3 = df_long["Error [mm]"].quantile(0.95)    #QUARTILE 2
    IQR = Q3 - Q1                                #InterQuartile Range
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Filter data
    df_long = df_long[(df_long["Error [mm]"] >= lower) & 
                    (df_long["Error [mm]"] <= upper)]
    # VIOLINPLOTS
    plt.figure(figsize=(15, 6))
    fig_name = f"Distribution of differences MLC bank {Y}.png"
    sns.violinplot(
        x="MLC", y="Error [mm]", 
        data=df_long, 
        inner="quartile", density_norm="width", cut=0
    )
    plt.xticks(rotation=90)
    plt.title(f"Distribution of differences per leaf of bank {Y} of the MLC")
    plt.xlabel("Leaf number")
    plt.ylabel("Position difference [mm]")
    plt.savefig(fig_name, dpi=300, bbox_inches="tight")
    plt.close()
    return {
        "image": fig_name
    }

In [25]:
######## VIOLINPLOT BANK Y1
V_PLOT_Y1 = D_MLC_VIO(Dif_Y1, "Y1")

In [26]:
######## VIOLINPLOT BANK Y2
V_PLOT_Y2 = D_MLC_VIO(Dif_Y2, "Y2")

In [27]:
####### AREA BY MLC
def Area(Y1_MLC, Y2_MLC):
    DIST = Y1_MLC.values - Y2_MLC.values
    AREA = abs((DIST*5).sum(axis=1)).tolist()
    return AREA 

In [28]:
######## AREAS
DCM_AREA = Area(DCM_Y1, DCM_Y2)
CSV_AREA = Area(CSV_Y1, CSV_Y2)

In [29]:
####### DIF. AREAS
def A_E(C_CP, AREA_DCM, AREA_CSV):
    DIF_A = np.array(AREA_DCM) - np.array(AREA_CSV)
    # SCATTER
    fig_name = "Difference of areas.png" 
    plt.figure(figsize=(15, 6))
    plt.scatter(C_CP, DIF_A)
    plt.xlabel("Control point")
    plt.ylabel("Difference of areas [mm^2]")
    plt.title("Areas difference per control point")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fig_name, dpi=300, bbox_inches="tight")
    plt.close()
    return{
        "image": fig_name
    }

In [30]:
######## ERROR AREAS
ERROR_A = A_E(CP_common, DCM_AREA, CSV_AREA)

In [31]:
######## DIF. POS. JAWS
Dif_X1 = DCM_X1.values - CSV_X1.values
Dif_X2 = DCM_X2.values - CSV_X2.values

In [32]:
######## DIF. JAWS
def E_J(C_CP, Dif_Mor, X):
    # STATISTICS
    stat = STAT(Dif_Mor)
    # SCATTER
    fig_name = f"Difference of jaw-{X}.png" 
    plt.figure(figsize=(15, 6))
    plt.scatter(C_CP, Dif_Mor)
    plt.xlabel("Control point")
    plt.ylabel("Average difference [mm]")
    plt.title(f"Jaw position difference-{X} per control point")
    plt.axhline(1, color='g', linestyle='--')
    plt.axhline(0, color='black', linestyle='--')
    plt.axhline(-1, color='g', linestyle='--')
    plt.grid(True)
    plt.savefig(fig_name, dpi=300, bbox_inches="tight")
    plt.close()
    # Messages
    messages_jaw = []
    messages_jaw.append(f"Mean value: {stat['Mean']:.5f} mm")
    messages_jaw.append(f"SD: {stat['SD']:.5f} mm")
    messages_jaw.append(f"P-value: {stat['P-value']:.5e}")
    ci_low, ci_high = stat["ci"]
    messages_jaw.append(f"IC 95%: [{ci_low:.5f}, {ci_high:.5f}] mm")
    
    return{
        "image": fig_name,
        "messages": messages_jaw
    }

In [33]:
######## JAW X1
E_X1 = E_J(CP_common, Dif_X1 , "X1")

In [34]:
######## JAW X2
E_X2 = E_J(CP_common, Dif_X2, "X2")

In [35]:
######### DIF. GANTRY ANGLE
Dif_G = DCM_GTY.values - CSV_GTY.values

In [36]:
######## DIF. GANTRY
def GE(CP_C, Dif_GANTRY):
    # STATISTIC
    stat = STAT(Dif_GANTRY)
    # SCATTER
    fig_name = "Difference of gantry.png"
    plt.figure(figsize=(15, 6))
    plt.scatter(CP_C, Dif_GANTRY)
    plt.xlabel("Control point")
    plt.ylabel("Average difference [grades]")
    plt.title("Angular position difference of the gantry per control point")
    plt.axhline(0, color='black', linestyle='--')
    plt.axhline(0.5, color='g', linestyle='--')
    plt.axhline(-0.5, color='g', linestyle='--')
    plt.grid(True)
    plt.savefig(fig_name, dpi=300, bbox_inches="tight")
    plt.close()
    # Messages
    messages_G = []
    messages_G.append(f"Mean value: {stat['Mean']:.5f} mm")
    messages_G.append(f"Desv. Std: {stat['SD']:.5f} mm")
    messages_G.append(f"P-value: {stat['P-value']:.5e}")
    ci_low, ci_high = stat["ci"]
    messages_G.append(f"IC 95%: [{ci_low:.5f}, {ci_high:.5f}] mm")
    
    return{
        "image": fig_name,
        "messages": messages_G
    }

In [37]:
######## GANTRY
E_G = GE(CP_common, Dif_G)

In [38]:
######## MLC SPEED
def spd_mlc(Y, XY_M, T_M):
    Dif_pos = abs(XY_M.diff().fillna(0))
    Dif_time = T_M.diff().fillna(0.04)
    SPD = Dif_pos.values/Dif_time.values[:, np.newaxis]
    SPD_Mean = SPD.mean(axis=0)
    SPD_M = np.round(SPD_Mean.mean(),2)
    SPD_MAX = np.round(SPD_Mean.max(),2)
    N_L = XY_M.shape[1] + 1
    N = np.arange(1, N_L)
    # STATISTICS
    stat = STAT(SPD_Mean)
    # SCATTER
    fig_name = f"Average speed {Y}.png"
    plt.figure(figsize=(15, 6))
    plt.scatter(N, SPD_Mean)
    plt.xticks(rotation=90)
    plt.xlabel("Leaf number")
    plt.ylabel("Average speed [mm/s]")
    plt.title(f"Average speed per leaf of bank {Y} of the MLC")
    plt.xticks(N)
    plt.grid(True)
    plt.savefig(fig_name, dpi=300, bbox_inches="tight")
    plt.close()
    # Messages
    messages_SPD = []
    messages_SPD.append(f"Average speed of bank {Y}: {SPD_M} mm/s")
    messages_SPD.append(f"Max. speed of bank {Y}: {SPD_MAX} mm/s")
    messages_SPD.append(f"SD: {stat['SD']:.5f} mm")
    messages_SPD.append(f"P-value: {stat['P-value']:.5e}")
    
    return {
        "image": fig_name,
        "messages": messages_SPD,
    }

In [39]:
######## SPEED BANK Y1
SPD_Y1 = spd_mlc("Y1", Y1_MOVE, TIME_MOV)

In [40]:
######## SPEED BANK Y2
SPD_Y2 = spd_mlc("Y2", Y2_MOVE, TIME_MOV)

In [41]:
######## EFFECTIVE TIME
def T_Ef(TIME_T, TIME_EF, P_T):
    T_T = round(TIME_T/60, 4)
    T_E = round(TIME_EF/60, 4)
    messages_T = []
    messages_T.append(
            f"Total session time: {T_T} min"
        )
    messages_T.append(
            f"Effective session time: {T_E} min"
    )
    messages_T.append(
            f"Percentage of effective time per session: {P_T} %"
    )
    return {
        "messages": messages_T
    }

In [42]:
######## EFFECTIVE SESSION TIME
TIME_EF = T_Ef(CSV_TIME_TOTAL, CSV_TIME_EF, CSV_P_TOTAL)

In [43]:
######## MODULATION COMPLEXITY SCORE
def Dist(I_MLC, D_MLC):
    DIST = D_MLC.values - I_MLC.values
    return DIST

# Aperture Area Variability (AAV)
def AAV_i(DIF_MLC):
    NMLC = DIF_MLC.shape[1]
    S_Ap = DIF_MLC.sum(axis=1)
    Max_Ap = np.max(DIF_MLC, axis=1)*NMLC
    Max_Ap[Max_Ap == 0] = np.nan
    AAV_list = np.array(S_Ap/Max_Ap)
    return AAV_list

# Leaf Sequence Variability (LSV)
def LSV_i(POS_MLC):
    NF = POS_MLC.shape[1]
    Dif_F = np.abs(np.diff(POS_MLC.values, axis=1))
    P_max = np.max(np.abs(POS_MLC))
    if P_max == 0:
        return np.full(POS_MLC.shape[0], np.nan)
    LSV_list = np.array(np.sum(P_max - Dif_F, axis=1)/((NF - 1)* P_max))
    return LSV_list

# Modulation Complexity Score (MCS)
def MCS(AAV, LSV, MU, Y):
    MCS_s = 0
    mu = np.array(MU)
    MU_T = np.sum(MU)
    if MU_T == 0:
        return np.nan
    for i in range(len(MU) - 1):
        aav_avg = (AAV[i] + AAV[i+1])/2
        lsv_avg = (LSV[i] + LSV[i+1])/2
        mu_r = (mu[i])/MU_T
        MCS_s += aav_avg*lsv_avg*mu_r
    messages_MCS = []
    messages_MCS.append(
            f"Modulation complexity score ''MCS'' (bank {Y})= {round(MCS_s, 4)}"
    )
    return {
        "MCS_val": MCS_s,
        "messages" : messages_MCS
    }

In [44]:
######## METRIC AAV
CSV_DIST = Dist(CSV_Y1_MLC, CSV_Y2_MLC)
AAV_CSV = AAV_i(CSV_DIST)

In [45]:
######## MCS BANK Y1
LSV_CSV_Y1 = LSV_i(CSV_Y1_MLC)
MCS_CSV_Y1 = MCS(AAV_CSV, LSV_CSV_Y1, CSV_D_CP, "Y1")

In [46]:
######## MCS BANK Y2
LSV_CSV_Y2 = LSV_i(CSV_Y2_MLC)
MCS_CSV_Y2 = MCS(AAV_CSV, LSV_CSV_Y2, CSV_D_CP, "Y2")

In [47]:
######## MCS (MEAN VALUE)
def MCS_M(MCS_1, MCS_2):
    MCS_MEAN = round((MCS_1["MCS_val"] + MCS_2["MCS_val"])/2, 5)
    message_MCS = []
    message_MCS.append(
            f"Mean value of modulation complexity score ''MCS''= {round(MCS_MEAN, 5)}"
    )
    return{
        "messages": message_MCS
    }

In [48]:
######## MCS
MCS_Mean = MCS_M(MCS_CSV_Y1, MCS_CSV_Y2)

In [49]:
######## PDF SAVE WINDOW
tk.Tk().withdraw()
PDF_name = f"{ID}_PSQA.pdf"
Window_root_PDF = askdirectory(title="Select folder")
root_PDF = os.path.join(
    Window_root_PDF,
    PDF_name
)

In [50]:
######## PDF GENERATION
doc = SimpleDocTemplate(
    root_PDF,
    pagesize=A4,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40
)
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    name="CENTER",
    parent = styles["Heading3"],
    alignment = TA_CENTER
    )
)
blocks_pdf = []

In [51]:
######## SCALE IMAGES
width = 420

######## MLC Y1
img_Y1 = RLImage(E_Y1["image"])
w, h = img_Y1.drawWidth, img_Y1.drawHeight
img_Y1.drawWidth = width
img_Y1.drawHeight = h * (width / w)
img_Y1.hAlign = "CENTER"

######## MLC Y2
img_Y2 = RLImage(E_Y2["image"])
w, h = img_Y2.drawWidth, img_Y2.drawHeight
img_Y2.drawWidth = width
img_Y2.drawHeight = h * (width / w)
img_Y2.hAlign = "CENTER"

######## VIOLINPLOT Y1
Dist_Y1 = RLImage(V_PLOT_Y1["image"])
w, h = Dist_Y1.drawWidth, Dist_Y1.drawHeight
Dist_Y1.drawWidth = width
Dist_Y1.drawHeight = h * (width / w)
Dist_Y1.hAlign = "CENTER"

######## VIOLINPLOT Y2
Dist_Y2 = RLImage(V_PLOT_Y2["image"])
w, h = Dist_Y2.drawWidth, Dist_Y2.drawHeight
Dist_Y2.drawWidth = width
Dist_Y2.drawHeight = h * (width / w)
Dist_Y2.hAlign = "CENTER"

######## Jaw X1
Jaw_X1 = RLImage(E_X1["image"])
w, h = Jaw_X1.drawWidth, Jaw_X1.drawHeight
Jaw_X1.drawWidth = width
Jaw_X1.drawHeight = h * (width / w)
Jaw_X1.hAlign = "CENTER"

######## Jaw X2
Jaw_X2 = RLImage(E_X2["image"])
w, h = Jaw_X2.drawWidth, Jaw_X2.drawHeight
Jaw_X2.drawWidth = width
Jaw_X2.drawHeight = h * (width / w)
Jaw_X2.hAlign = "CENTER"

######## Area
img_A = RLImage(ERROR_A["image"])
w, h = img_A.drawWidth, img_A.drawHeight
img_A.drawWidth = width
img_A.drawHeight = h * (width / w)
img_A.hAlign = "CENTER"

######## Gantry
img_G = RLImage(E_G["image"])
w, h = img_G.drawWidth, img_G.drawHeight
img_G.drawWidth = width
img_G.drawHeight = h * (width/ w)
img_G.hAlign = "CENTER"

######## SPD Y1
img_VY1 = RLImage(SPD_Y1["image"])
w, h = img_VY1.drawWidth, img_VY1.drawHeight
img_VY1.drawWidth = width
img_VY1.drawHeight = h * (width / w)
img_VY1.hAlign = "CENTER"

######## SPD Y2
img_VY2 = RLImage(SPD_Y2["image"])
w, h = img_VY2.drawWidth, img_VY2.drawHeight
img_VY2.drawWidth = width
img_VY2.drawHeight = h * (width / w)
img_VY2.hAlign = "CENTER"

In [52]:
######## TITLE
blocks_pdf.append(
    Paragraph("Mechanical analysis report for SBRT-PSQA", styles["Title"])
)
blocks_pdf.append(Spacer(1, 20))

In [53]:
######## HEADER
blocks_pdf.append(
    Paragraph(f"Patient name: {Patient}", styles["Normal"])
)
blocks_pdf.append(Spacer(1, 5))

blocks_pdf.append(
    Paragraph(f"Patient ID: {ID}", styles["Normal"])
)
blocks_pdf.append(Spacer(1, 5))

blocks_pdf.append(
    Paragraph(f"Date and time: {Date}", styles["Normal"])
)
blocks_pdf.append(Spacer(1, 5))

blocks_pdf.append(
    Paragraph(f"Target: {Target}", styles["Normal"])
)
blocks_pdf.append(Spacer(1, 5))

In [54]:
######## MONITOR UNITS
blocks_pdf.append(
    Paragraph("Monitor units analysis", styles["Heading2"])
)
blocks_pdf.append(Spacer(1, 5))

for msg in E_D["messages"]:
    blocks_pdf.append(Paragraph(msg, styles["Normal"]))
    blocks_pdf.append(Spacer(1, 5))

In [55]:
######## ERROR MLC
blocks_pdf.append(
    Paragraph("Average position difference of MLC leaves", styles["Heading2"])
)
blocks_pdf.append(Spacer(1, 5))

blocks_pdf.append(
    KeepTogether([
        Paragraph("Bank Y-1", styles["CENTER"]),
        Spacer(1, 10),
        img_Y1,
        Spacer(1, 15),      
        # Block of messages
        Paragraph("<b>Bank limits Y1</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in E_Y1["messages_pos"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ],    
        # Block statistics
        Paragraph("<b>Statistics</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in E_Y1["messages_std"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ]
    ])
)

blocks_pdf.append(
    KeepTogether([
        Paragraph("Bank Y-2", styles["CENTER"]),
        Spacer(1, 10),
        img_Y2,
        Spacer(1, 15),
        # Block of messages
        Paragraph("<b>Bank limits Y2</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in E_Y2["messages_pos"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 10))
        ],
        # Block statistics
        Paragraph("<b>Statistics</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in E_Y2["messages_std"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ]
    ])
)

In [56]:
######## DIST ERROR MLC
blocks_pdf.append(
    Paragraph("Distribution of positional differences of MLC leaves", styles["Heading2"])
)
blocks_pdf.append(Spacer(1, 5))

blocks_pdf.append(
    KeepTogether([
        Paragraph("Bank Y-1", styles["CENTER"]),
        Spacer(1, 5),
        Dist_Y1,
        Spacer(1, 5)
    ])
)

blocks_pdf.append(
    KeepTogether([
        Paragraph("Bank Y-2", styles["CENTER"]),
        Spacer(1, 5),
        Dist_Y2,
        Spacer(1, 5)
    ])
)

In [57]:
######## ERROR JAWS
blocks_pdf.append(
    Paragraph("Jaw position difference per control point", styles["Heading2"])
)
blocks_pdf.append(Spacer(1, 5))

blocks_pdf.append(
    KeepTogether([
        Paragraph("Jaw X-1", styles["CENTER"]),
        Spacer(1, 10),
        Jaw_X1,
        Spacer(1, 15),
        # Block statistics
        Paragraph("<b>Statistics</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in E_X1["messages"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ]
    ])
)

blocks_pdf.append(
    KeepTogether([
        Paragraph("Jaw X-2", styles["CENTER"]),
        Spacer(1, 10),
        Jaw_X2,
        Spacer(1, 15),
        # Block statistics
        Paragraph("<b>Statistics</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in E_X2["messages"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ]
    ])
)

In [58]:
######## ERROR ÁREA
blocks_pdf.append(
    KeepTogether([
        Paragraph("Difference in areas generated by the MLC per control point", styles["Heading2"]),
        Spacer(1, 5),
        img_A,
        Spacer(1, 5)
    ])
)

In [59]:
######## ERROR GANTRY
blocks_pdf.append(
    KeepTogether([
        Paragraph("Gantry angular position difference per control point", styles["Heading2"]),
        Spacer(1, 10),
        img_G,
        Spacer(1, 15),
        # Block statistics
        Paragraph("<b>Statistics</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in E_G["messages"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ]
    ])
)

In [60]:
######## SPEED MLC
blocks_pdf.append(
    Paragraph("Spees per MLC leaf", styles["Heading2"])
)
blocks_pdf.append(Spacer(1, 5))

blocks_pdf.append(
    KeepTogether([
        Paragraph("Bank Y-1", styles["CENTER"]),
        Spacer(1, 10),
        img_VY1,
        Spacer(1, 15),
        # Block statistics
        Paragraph("<b>Statistics</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in SPD_Y1["messages"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ]
    ])
)

blocks_pdf.append(
    KeepTogether([
        Paragraph("Bank Y-2", styles["CENTER"]),
        Spacer(1, 10),
        img_VY2,
        Spacer(1, 15),
        # Block statistics
        Paragraph("<b>Statistics</b>", styles["Normal"]),
        Spacer(1, 5),
        *[
            item 
            for msg in SPD_Y2["messages"]
            for item in (Paragraph(msg, styles["Normal"]), Spacer(1, 5))
        ]
    ])
)

In [61]:
######## EFFECTIVE TIME
blocks_pdf.append(
    Paragraph("Effective session time", styles["Heading2"])
)
blocks_pdf.append(Spacer(1, 5))

for msg in TIME_EF["messages"]:
    blocks_pdf.append(Paragraph(msg, styles["Normal"]))
    blocks_pdf.append(Spacer(1, 5))

In [62]:
######## MCS
blocks_pdf.append(
    Paragraph("Modulation Complexity Score", styles["Heading2"])
)
blocks_pdf.append(Spacer(1, 5))

for msg in MCS_CSV_Y1["messages"]:
    blocks_pdf.append(Paragraph(msg, styles["Normal"]))
    blocks_pdf.append(Spacer(1, 5))

for msg in MCS_CSV_Y2["messages"]:
    blocks_pdf.append(Paragraph(msg, styles["Normal"]))
    blocks_pdf.append(Spacer(1, 5))

for msg in MCS_Mean["messages"]:
    blocks_pdf.append(Paragraph(msg, styles["Normal"]))
    blocks_pdf.append(Spacer(1, 5))

In [63]:
######## BUILD PDF
doc.build(blocks_pdf)